# 07 — Local DICOM sample audit

Notebook này chạy **trên Windows/local**, sau khi tải `demo_sample.zip` từ Kaggle về.
Nó chỉ kiểm tra package, metadata, DICOM headers, thứ tự slices và decode thử một ảnh mỗi series.
Chưa chạy preprocessing của ViT/DINOv2 và chưa load checkpoint.

## Expected input

`demo/input/demo_sample.zip` hoặc thư mục đã giải nén chứa `manifest.json`.

Chạy từ project root trong VS Code/Jupyter để notebook tự tìm `pyproject.toml`.

In [11]:
from pathlib import Path
import sys
import json
import pandas as pd
from IPython.display import display

def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / 'pyproject.toml').is_file():
            return candidate
    raise FileNotFoundError('Không tìm thấy pyproject.toml. Hãy mở notebook từ project root.')

PROJECT_ROOT = find_project_root(Path.cwd())
SRC_ROOT = PROJECT_ROOT / 'src'
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from rsna_knee.dicom.sample import (
    audit_demo_sample, load_demo_sample, write_audit_report,
)

SAMPLE_ZIP = PROJECT_ROOT / 'demo' / 'input' / 'demo_sample.zip'
SAMPLE_DIR = PROJECT_ROOT / 'demo' / 'input' / 'demo_sample'
OUTPUT_JSON = PROJECT_ROOT / 'demo' / 'output' / 'sample_audit.json'

if SAMPLE_ZIP.is_file():
    SAMPLE_SOURCE = SAMPLE_ZIP
elif SAMPLE_DIR.is_dir():
    SAMPLE_SOURCE = SAMPLE_DIR
else:
    raise FileNotFoundError(
        f'Không tìm thấy sample. Đặt ZIP tại {SAMPLE_ZIP} hoặc giải nén vào {SAMPLE_DIR}.'
    )

print('Project root:', PROJECT_ROOT)
print('Sample source:', SAMPLE_SOURCE)
print('Audit output:', OUTPUT_JSON)

Project root: C:\Users\THINH DINH\Desktop\space\rsna-knee-abnormality-detection
Sample source: C:\Users\THINH DINH\Desktop\space\rsna-knee-abnormality-detection\demo\input\demo_sample
Audit output: C:\Users\THINH DINH\Desktop\space\rsna-knee-abnormality-detection\demo\output\sample_audit.json


In [13]:
from pathlib import Path, PurePosixPath
import zipfile
import shutil

ZIP_PATH = Path.home() / "Downloads" / "demo_sample.zip"

# Đặt dữ liệu ở thư mục có đường dẫn ngắn
SHORT_ROOT = Path(r"C:\rsna_demo")
SHORT_ROOT.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(ZIP_PATH, "r") as z:
    assert z.testzip() is None, "ZIP corrupted!"

    dicom_files = [
        n for n in z.namelist()
        if "/dicom/" in n.lower()
        and n.lower().endswith(".dcm")
    ]

    assert len(dicom_files) == 164

    # Kiểm tra đường dẫn ZIP
    for name in z.namelist():
        p = PurePosixPath(name)
        assert not p.is_absolute()
        assert ".." not in p.parts

    print("Extracting", len(dicom_files), "DICOM files...")

    z.extractall(SHORT_ROOT)

# Cập nhật SAMPLE_SOURCE sang vị trí mới
SAMPLE_SOURCE = SHORT_ROOT / "demo_sample"

extracted = list(
    (SAMPLE_SOURCE / "dicom").rglob("*.dcm")
)

print("Extraction completed!")
print("New sample root:", SAMPLE_SOURCE)
print("DICOM files:", len(extracted))

assert len(extracted) == 164

Extracting 164 DICOM files...
Extraction completed!
New sample root: C:\rsna_demo\demo_sample
DICOM files: 164


## 1. Load sample package

In [15]:
manifest = json.loads(
    (SAMPLE_SOURCE / "manifest.json").read_text(encoding="utf-8")
)

dicom_root = (
    SAMPLE_SOURCE
    / "dicom"
    / manifest["source"]["study_instance_uid"]
)

dicom_files = list(dicom_root.rglob("*.dcm"))

print("DICOM root:", dicom_root)
print("DICOM files:", len(dicom_files))
print("Expected:", manifest["dicom"]["file_count"])

assert len(dicom_files) == 164

DICOM root: C:\rsna_demo\demo_sample\dicom\1.2.826.0.1.3680043.8.498.10095687747295410396510538520594649149
DICOM files: 164
Expected: 164


In [16]:
study = load_demo_sample(SAMPLE_SOURCE)

print('StudyInstanceUID:', study.study_uid)
print('Sample root:', study.root)
print('Number of selected series:', len(study.series))
print('Number of DICOM files:', sum(s.n_files for s in study.series.values()))

display(study.metadata)
display(study.selection)

StudyInstanceUID: 1.2.826.0.1.3680043.8.498.10095687747295410396510538520594649149
Sample root: C:\rsna_demo\demo_sample
Number of selected series: 5
Number of DICOM files: 164


,StudyInstanceUID,SeriesInstanceUID,Fluid_Sensitive,Fat_Suppression,Anatomical_Plane,selected_for,dicom_relative_dir
0,1.2.826.0.1.3680043.8.498.10095687747295410396...,1.2.826.0.1.3680043.8.498.10791858932231443712...,0,0,Sagittal,DINOv2:SAG_NONFLUID,dicom/1.2.826.0.1.3680043.8.498.10095687747295...
1,1.2.826.0.1.3680043.8.498.10095687747295410396...,1.2.826.0.1.3680043.8.498.12451507871565834868...,0,0,Coronal,DINOv2:COR_NONFLUID,dicom/1.2.826.0.1.3680043.8.498.10095687747295...
2,1.2.826.0.1.3680043.8.498.10095687747295410396...,1.2.826.0.1.3680043.8.498.21497837050858589852...,1,1,Coronal,DINOv2:COR_FLUID;ViT:Coronal,dicom/1.2.826.0.1.3680043.8.498.10095687747295...
3,1.2.826.0.1.3680043.8.498.10095687747295410396...,1.2.826.0.1.3680043.8.498.37372672584387115708...,1,1,Sagittal,DINOv2:SAG_FLUID;ViT:Sagittal,dicom/1.2.826.0.1.3680043.8.498.10095687747295...
4,1.2.826.0.1.3680043.8.498.10095687747295410396...,1.2.826.0.1.3680043.8.498.67495435033389756963...,1,1,Axial,DINOv2:AX_FLUID;ViT:Axial,dicom/1.2.826.0.1.3680043.8.498.10095687747295...


,model,role_or_plane,SeriesInstanceUID,selection_source
0,DINOv2,SAG_FLUID,1.2.826.0.1.3680043.8.498.37372672584387115708...,primary_cache_equivalent
1,DINOv2,COR_FLUID,1.2.826.0.1.3680043.8.498.21497837050858589852...,primary_cache_equivalent
2,DINOv2,AX_FLUID,1.2.826.0.1.3680043.8.498.67495435033389756963...,primary_cache_equivalent
3,DINOv2,SAG_NONFLUID,1.2.826.0.1.3680043.8.498.10791858932231443712...,supplementary_fallback
4,DINOv2,COR_NONFLUID,1.2.826.0.1.3680043.8.498.12451507871565834868...,supplementary_fallback
5,ViT-B/16,Sagittal,1.2.826.0.1.3680043.8.498.37372672584387115708...,highest_fluid_fat_priority_per_plane
6,ViT-B/16,Coronal,1.2.826.0.1.3680043.8.498.21497837050858589852...,highest_fluid_fat_priority_per_plane
7,ViT-B/16,Axial,1.2.826.0.1.3680043.8.498.67495435033389756963...,highest_fluid_fat_priority_per_plane


## 2. Audit DICOM headers and decode one slice per series

Pixel decode có thể báo lỗi nếu máy local thiếu codec cho một transfer syntax nén. Nếu lỗi xuất hiện, giữ nguyên thông tin lỗi để xử lý ở bước sau; không âm thầm bỏ qua.

In [17]:
report = audit_demo_sample(study, decode_pixels=True)

print('Status:', report['status'])
print('DINO roles:', f"{report['dino_roles_selected']}/5")
print('ViT planes:', f"{report['vit_planes_selected']}/3")
print('Series:', report['series_count'])
print('DICOM files:', report['dicom_file_count'])
print('Warnings:', report['warning_count'], '| Errors:', report['error_count'])

display(pd.DataFrame(report['series']))
display(pd.DataFrame(report['issues']))

Status: ok
DINO roles: 5/5
ViT planes: 3/3
Series: 5
DICOM files: 164
Warnings: 0 | Errors: 0


,SeriesInstanceUID,plane,fluid_sensitive,fat_suppression,n_files,valid_headers,sort_mode,selected_for,pixel_decode,pixel_shape
0,1.2.826.0.1.3680043.8.498.10791858932231443712...,Sagittal,0,0,36,36,ImagePositionPatient_projected,DINOv2:SAG_NONFLUID,ok,"[400, 400]"
1,1.2.826.0.1.3680043.8.498.12451507871565834868...,Coronal,0,0,30,30,ImagePositionPatient_projected,DINOv2:COR_NONFLUID,ok,"[384, 384]"
2,1.2.826.0.1.3680043.8.498.21497837050858589852...,Coronal,1,1,30,30,ImagePositionPatient_projected,DINOv2:COR_FLUID;ViT-B/16:Coronal,ok,"[320, 320]"
3,1.2.826.0.1.3680043.8.498.37372672584387115708...,Sagittal,1,1,36,36,ImagePositionPatient_projected,DINOv2:SAG_FLUID;ViT-B/16:Sagittal,ok,"[320, 320]"
4,1.2.826.0.1.3680043.8.498.67495435033389756963...,Axial,1,1,32,32,ImagePositionPatient_projected,DINOv2:AX_FLUID;ViT-B/16:Axial,ok,"[320, 320]"


""


## 3. Check model-to-series mapping

In [18]:
dino_roles = ['SAG_FLUID', 'COR_FLUID', 'AX_FLUID', 'SAG_NONFLUID', 'COR_NONFLUID']
vit_planes = ['Sagittal', 'Coronal', 'Axial']

mapping_rows = []
for role in dino_roles:
    item = study.series_for_dino_role(role)
    mapping_rows.append({'model': 'DINOv2', 'role_or_plane': role,
                        'SeriesInstanceUID': item.series_uid if item else '',
                        'n_slices': item.n_files if item else 0})
for plane in vit_planes:
    item = study.series_for_vit_plane(plane)
    mapping_rows.append({'model': 'ViT-B/16', 'role_or_plane': plane,
                        'SeriesInstanceUID': item.series_uid if item else '',
                        'n_slices': item.n_files if item else 0})

mapping_df = pd.DataFrame(mapping_rows)
display(mapping_df)

,model,role_or_plane,SeriesInstanceUID,n_slices
0,DINOv2,SAG_FLUID,1.2.826.0.1.3680043.8.498.37372672584387115708...,36
1,DINOv2,COR_FLUID,1.2.826.0.1.3680043.8.498.21497837050858589852...,30
2,DINOv2,AX_FLUID,1.2.826.0.1.3680043.8.498.67495435033389756963...,32
3,DINOv2,SAG_NONFLUID,1.2.826.0.1.3680043.8.498.10791858932231443712...,36
4,DINOv2,COR_NONFLUID,1.2.826.0.1.3680043.8.498.12451507871565834868...,30
5,ViT-B/16,Sagittal,1.2.826.0.1.3680043.8.498.37372672584387115708...,36
6,ViT-B/16,Coronal,1.2.826.0.1.3680043.8.498.21497837050858589852...,30
7,ViT-B/16,Axial,1.2.826.0.1.3680043.8.498.67495435033389756963...,32


## 4. Save audit result

Chỉ chuyển sang port preprocessing khi package audit không có lỗi. Warning cần được đọc và giải thích trước khi bỏ qua.

In [19]:
saved_path = write_audit_report(report, OUTPUT_JSON)
print('Saved:', saved_path)

if report['status'] != 'ok':
    raise RuntimeError('Sample audit có lỗi. Xử lý các errors trước khi triển khai preprocessing.')

print('DICOM sample audit passed. Ready for model-specific preprocessing port.')

Saved: C:\Users\THINH DINH\Desktop\space\rsna-knee-abnormality-detection\demo\output\sample_audit.json
DICOM sample audit passed. Ready for model-specific preprocessing port.
